# Model Training, MLflow Logging, & Statistical A/B Testing

**Objective:**
Train a *machine learning* model to detect *fraud*, log all parameters and metrics to the *Model Registry* (*MLOps*), and conduct statistical significance tests (A/B testing) to validate the model’s performance.

**Analytical Approach:**
1. **Experiment Tracking:** Prevent the loss of experiment metadata by automating logging using MLflow.
2. **Algorithmic Contenders (A/B Setup):** Train two leading *Gradient Boosting* algorithms (LightGBM as the *Baseline/Control* and XGBoost as the *Challenger/Variant*).
3. **Statistical Rigor (McNemar’s Test):** Comparing the predictions of both models on the same *test set*. In statistics and computer science, comparing AUC scores alone is not sufficient; we must mathematically prove (with a *p-value* < 0.05) that the difference in performance between the two models is statistically significant, rather than due to random variance.
4. **Threshold Optimization:** Optimizing the probability threshold (not the default 0.5) to balance *False Positives* and *False Negatives* according to business needs.

In [1]:
import pandas as pd
import numpy as np
import warnings
import mlflow
import mlflow.xgboost
import mlflow.lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score, 
    precision_recall_curve, confusion_matrix, f1_score
)
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings('ignore')

# 1. Konfigurasi MLflow ke Local Docker Server
MLFLOW_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("Risk_Fraud_Detection")
print(f"MLflow terhubung ke: {MLFLOW_URI}")

# 2. Memuat Data Preprocessed (Hasil dari Notebook 02)
train_df = pd.read_parquet('../dataset/02_processed/train_processed.parquet')
test_df = pd.read_parquet('../dataset/02_processed/test_processed.parquet')

X_train = train_df.drop(columns=['is_fraud'])
y_train = train_df['is_fraud']
X_test = test_df.drop(columns=['is_fraud'])
y_test = test_df['is_fraud']

print(f"Data Loaded. X_train: {X_train.shape}, X_test: {X_test.shape}")

/home/bayua/miniconda3/envs/py310/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2026/10/06 07:37:51 INFO mlflow.tracking.fluent: Experiment with name 'Risk_Fraud_Detection' does not exist. Creating a new experiment.


MLflow terhubung ke: http://localhost:5000
Data Loaded. X_train: (1037340, 22), X_test: (259335, 22)


## Phase 1: Training Model A - Baseline (LightGBM)

**Problem:**
We need a very fast and memory-efficient *baseline* model as a reference point (Model A / *Control Group* in A/B Testing).

**Solution:**
We use *LightGBM* optimized for large datasets with a class weighting parameter (`scale_pos_weight`) to handle the minority class ratio (0.58%). All artifacts are logged to MLflow.

In [2]:
# Menghitung rasio imbalance untuk pembobotan
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# Memulai sesi MLflow untuk Model A
with mlflow.start_run(run_name="Model_A_LightGBM_Baseline"):
    # Definisi & Training Model
    lgbm_model = LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        scale_pos_weight=pos_weight,
        random_state=42,
        n_jobs=-1
    )
    lgbm_model.fit(X_train, y_train)
    
    # Prediksi
    y_pred_proba_a = lgbm_model.predict_proba(X_test)[:, 1]
    y_pred_a = lgbm_model.predict(X_test)
    
    # Kalkulasi Metrik (Fokus pada PR-AUC karena dataset sangat imbalanced)
    roc_auc_a = roc_auc_score(y_test, y_pred_proba_a)
    pr_auc_a = average_precision_score(y_test, y_pred_proba_a)
    f1_a = f1_score(y_test, y_pred_a)
    
    # Logging ke MLflow
    mlflow.log_params(lgbm_model.get_params())
    mlflow.log_metrics({"roc_auc": roc_auc_a, "pr_auc": pr_auc_a, "f1_score": f1_a})
    mlflow.lightgbm.log_model(lgbm_model, "model_baseline_lgbm")
    
    print(f"[Model A] PR-AUC: {pr_auc_a:.4f} | ROC-AUC: {roc_auc_a:.4f} | F1-Score: {f1_a:.4f}")

[LightGBM] [Info] Number of positive: 6005, number of negative: 1031335
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006285 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1412
[LightGBM] [Info] Number of data points in the train set: 1037340, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005789 -> initscore=-5.146017
[LightGBM] [Info] Start training from score -5.146017


2026/10/06 07:37:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/06 07:38:09 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


[Model A] PR-AUC: 0.9258 | ROC-AUC: 0.9983 | F1-Score: 0.4766
🏃 View run Model_A_LightGBM_Baseline at: http://localhost:5000/#/experiments/1/runs/d1b9e8809b224c4b8cb713848d2f9da9
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Phase 2: Training Model B - Challenger (XGBoost)

**Problem:**
The *baseline model* may fail to capture the more complex nonlinear interactions among the geospatial and demographic features we have engineered.

**Solution:**
Build an XGBoost architecture (Model B / *Variant Group*) with more aggressive tree depth control.

In [3]:
# Memulai sesi MLflow untuk Model B
with mlflow.start_run(run_name="Model_B_XGBoost_Challenger"):
    # Definisi & Training Model
    xgb_model = XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=pos_weight,
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    
    # Prediksi
    y_pred_proba_b = xgb_model.predict_proba(X_test)[:, 1]
    y_pred_b = xgb_model.predict(X_test)
    
    # Kalkulasi Metrik
    roc_auc_b = roc_auc_score(y_test, y_pred_proba_b)
    pr_auc_b = average_precision_score(y_test, y_pred_proba_b)
    f1_b = f1_score(y_test, y_pred_b)
    
    # Logging ke MLflow
    mlflow.log_params(xgb_model.get_params())
    mlflow.log_metrics({"roc_auc": roc_auc_b, "pr_auc": pr_auc_b, "f1_score": f1_b})
    mlflow.xgboost.log_model(xgb_model, "model_challenger_xgb")
    
    print(f"[Model B] PR-AUC: {pr_auc_b:.4f} | ROC-AUC: {roc_auc_b:.4f} | F1-Score: {f1_b:.4f}")

2026/10/06 07:38:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[Model B] PR-AUC: 0.9001 | ROC-AUC: 0.9982 | F1-Score: 0.4228
🏃 View run Model_B_XGBoost_Challenger at: http://localhost:5000/#/experiments/1/runs/8f5c65539d954a9c9afbd34d76d3d504
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Phase 3: Statistical A/B Testing (McNemar’s Test)

**Problem:**
If Model B has a PR-AUC or F1 score that is slightly higher than Model A, can we immediately replace the old model in the *production* system? According to software engineering and statistical standards, differences in scores may simply be due to random chance.

**Solution:**
Use **McNemar’s Test**. This nonparametric statistical test is the gold standard in the *Machine Learning* literature (including PhD journals) for comparing two *classifiers* on the same *test set*.
*   **H0 (Null Hypothesis):** Model A and Model B have equivalent classification performance. (Equivalent error rates).
*   **H1 (Alternative Hypothesis):** The performance of the two models differs statistically significantly.

If the *p-value* is < 0.05, we reject H0 and empirically prove that one model is indeed superior.

In [4]:
# Membuat Tabel Kontingensi untuk McNemar's Test
# Cell [0,0]: Kedua model benar.
# Cell [0,1]: Model A benar, Model B salah.
# Cell [1,0]: Model A salah, Model B benar.
# Cell [1,1]: Kedua model salah.

correct_a = (y_pred_a == y_test)
correct_b = (y_pred_b == y_test)

table = pd.crosstab(correct_a, correct_b)
print("Contingency Table (True/False):")
display(table)

# Menjalankan McNemar's Test dengan Koreksi Kontinuitas Edwards
result = mcnemar(table, exact=False, correction=True)

print(f"\nMcNemar's Test Statistic: {result.statistic:.4f}")
print(f"P-Value: {result.pvalue:.4e}")

alpha = 0.05
if result.pvalue < alpha:
    print("\n[STATISTICAL CONCLUSION]: Significant!")
    print("Reject H0. There is a statistically significant difference in performance between LightGBM and XGBoost.")
else:
    print("\n[STATISTICAL CONCLUSION]: Not Significant.")
    print("Fail to reject H0. The difference in scores between the two models is most likely due to random variance.")

Contingency Table (True/False):


is_fraud,False,True
is_fraud,,
False,2475,763
True,1522,254575



McNemar's Test Statistic: 251.4503
P-Value: 1.2539e-56

[STATISTICAL CONCLUSION]: Significant!
Reject H0. There is a statistically significant difference in performance between LightGBM and XGBoost.


## Phase 3: Statistical A/B Testing (McNemar’s Test)

**Problem:**
If Model B has a PR-AUC or F1 score that is slightly higher than Model A, can we immediately replace the old model in the *production* system? According to software engineering and statistical standards, differences in scores may simply be due to random chance.

**Solution:**
Use **McNemar’s Test**. This nonparametric statistical test is the gold standard in the *Machine Learning* literature (including PhD journals) for comparing two *classifiers* on the same *test set*.
*   **H0 (Null Hypothesis):** Model A and Model B have equivalent classification performance. (Equivalent error rates).
*   **H1 (Alternative Hypothesis):** The performance of the two models differs statistically significantly.

If the *p-value* is < 0.05, we reject H0 and empirically prove that one model is indeed superior.

In [5]:
# Memilih probabilitas dari model pemenang (misal: XGBoost)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba_b)

# Menghitung F1 score untuk setiap threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
# Mengatasi pembagian dengan nol
f1_scores = np.nan_to_num(f1_scores)

# Mencari indeks dengan F1-Score tertinggi
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Default Threshold (0.5) F1-Score  : {f1_b:.4f}")
print(f"Optimal Threshold ({optimal_threshold:.4f}) F1-Score: {optimal_f1:.4f}")

# Menerapkan threshold optimal
y_pred_optimal = (y_pred_proba_b >= optimal_threshold).astype(int)

print("\nOperational Metrics Analysis (Classification Report)")
print(classification_report(y_test, y_pred_optimal))

Default Threshold (0.5) F1-Score  : 0.4228
Optimal Threshold (0.9741) F1-Score: 0.8371

Operational Metrics Analysis (Classification Report)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.91      0.77      0.84      1501

    accuracy                           1.00    259335
   macro avg       0.96      0.89      0.92    259335
weighted avg       1.00      1.00      1.00    259335

